# Target-gene UTR metrics: W per cluster, pseudobulk proportions, density (replaces 260423-utr_proportions_calculations)

For a curated set of target genes (top hits + mouse orthologs + ASD risk), compute the per-cluster proportions, bootstrap-CI W, per-cell W, raw UTR length distributions, and consensus peak positions. These are the inputs for the visualization notebooks (main_hits_pseudotime, mouse_orthologs, asd_risk_genes).

This notebook **drops the DRIMSeq calls** of the old `260423-utr_proportions_calculations.ipynb`. Target genes are now selected from the new `composite_ranking_optimized2_<lineage>.csv` (composite of LRT padj + delta_psi + rho_W).

Pipeline:

1. Load composite rankings per lineage.
2. Build the target gene set per lineage:
   - Top 30 by `composite_score_abs` (excluding orthologs and ASD; their hits are already covered).
   - 5 hand-picked mouse orthologs (HNRNPA2B1, QKI, GPM6B, KLC1, PEA15).
   - 36 hardcoded ASD risk genes (Li 2023 list).
3. Load the controls subset h5ads from `260504-effect_sizes_per_lineage.ipynb`.
4. For each (gene, lineage) in the target set, compute the five metrics.
5. Save CSVs and `gene_info.json` under `<utrcalling>/example_genes_results/`.

Inputs:

- `<utrcalling>/composite_ranking/composite_ranking_optimized2_<lineage>.csv`
- `<utrcalling>/effect_sizes/<lineage>_controls_raw_pt.h5ad`

Outputs (under `<utrcalling>/example_genes_results/`):

- `pseudobulk_proportions.csv`, `W_per_cluster.csv`, `W_per_cell.csv`,
  `utr_length_distributions.csv`, `consensus_peak_positions.csv`,
  `gene_info.json`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UTR")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt

from scipy.stats import rankdata
from statsmodels.stats.proportion import proportion_confint

warnings.filterwarnings("ignore")

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "font.size": 7,
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
    }
)
print("Setup complete.")

In [ ]:
LINEAGE_ORDER = ["Inhibitory neurons", "Excitatory neurons"]
LINEAGE_TAG = {"Inhibitory neurons": "inhibitory", "Excitatory neurons": "excitatory"}
LINEAGE_SLUGS = {l: t for l, t in LINEAGE_TAG.items()}

CANONICAL_CELLTYPE_BY_LINEAGE = {
    "Inhibitory neurons": ["vRG", "ccvRG", "INP", "LGE_IN", "CGE_LGE_IN", "CGE_IN"],
    "Excitatory neurons": ["oRG", "RG", "ccRG", "IP", "L6_CThPN", "L56", "L4", "L23"],
}

# Hardcoded gene lists (mirror old notebook)
MOUSE_ORTHOLOGS_HUMAN = ["HNRNPA2B1", "QKI", "GPM6B", "KLC1", "PEA15"]
MOUSE_TO_HUMAN_ORTHO = {
    "Hnrnpa2b1": "HNRNPA2B1",
    "Qk": "QKI",
    "Gpm6b": "GPM6B",
    "Klc1": "KLC1",
    "Pea15": "PEA15",
}

# Li 2023 ASD risk genes (the 36 perturbed in the CHOOSE paper)
ASD_GENES_LI2023 = [
    "ADNP",
    "ARID1B",
    "ASH1L",
    "ASXL3",
    "BAZ2B",
    "BCL11A",
    "CHD2",
    "CHD8",
    "CIC",
    "DDX3X",
    "DEAF1",
    "FOXP1",
    "ILF2",
    "IRF2BPL",
    "KAT2B",
    "KDM5B",
    "KDM6A",
    "KDM6B",
    "KMT2A",
    "KMT2C",
    "KMT5B",
    "LEO1",
    "MECP2",
    "MED13",
    "MED13L",
    "MYT1L",
    "PHF3",
    "POGZ",
    "SETD5",
    "SMARCC2",
    "SRCAP",
    "SRSF11",
    "TBL1XR1",
    "TBR1",
    "TCF20",
    "WAC",
]

N_TOP_HITS = 30
N_BOOTSTRAP = 2000
RANDOM_STATE = 42

BASEPATH = Path(f"{DATA_ROOT}/3utr/organoids/knoblich")
UTR_DIR = BASEPATH / "utrcalling"
COMPOSITE_DIR = UTR_DIR / "composite_ranking"
EFFECT_SIZES_DIR = UTR_DIR / "effect_sizes"
RESULTS_DIR = UTR_DIR / "example_genes_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Composite dir:  {COMPOSITE_DIR}")
print(f"Effect sizes:   {EFFECT_SIZES_DIR}")
print(f"Results dir:    {RESULTS_DIR}")

## Load composite rankings and define target genes


In [ ]:
composite_dfs = {}
for ltag in LINEAGE_TAG.values():
    p = COMPOSITE_DIR / f"composite_ranking_optimized2_{ltag}.csv"
    assert p.exists(), f"Missing composite ranking: {p}"
    composite_dfs[ltag] = pd.read_csv(p)
    print(f"{ltag}: {composite_dfs[ltag].shape}")

In [ ]:
target_genes_by_lineage = {}
top_hits_by_lineage = {}

# Genes to exclude from the "top hits" pick because they are covered by other tracks
exclude_set = set(MOUSE_ORTHOLOGS_HUMAN + ASD_GENES_LI2023)

for lname, ltag in LINEAGE_TAG.items():
    df = composite_dfs[ltag]
    top_pool = df[~df["gene_name"].isin(exclude_set)].copy()
    top_hits = top_pool.head(N_TOP_HITS)["gene_name"].tolist()
    top_hits_by_lineage[ltag] = top_hits

    # Union with orthologs and ASD list, preserving lineage labelling
    targets = sorted(set(top_hits + MOUSE_ORTHOLOGS_HUMAN + ASD_GENES_LI2023))
    target_genes_by_lineage[ltag] = targets

    print(f"\n[{ltag}]")
    print(
        f"  Top {N_TOP_HITS} by composite_score_abs (excluding orthologs+ASD): {len(top_hits)}"
    )
    print(f"  Mouse orthologs (5):       {len(MOUSE_ORTHOLOGS_HUMAN)}")
    print(f"  ASD risk genes (36):       {len(ASD_GENES_LI2023)}")
    print(f"  Target gene set (union):   {len(targets)}")

## Load controls subset h5ads


In [ ]:
adata_raw_by_lineage = {}
for ltag in LINEAGE_TAG.values():
    p = EFFECT_SIZES_DIR / f"{ltag}_controls_raw_pt.h5ad"
    assert (
        p.exists()
    ), f"Missing controls subset: {p}. Did you run notebook 1 (effect_sizes)?"
    adata_raw_by_lineage[ltag] = sc.read_h5ad(p)
    print(f"{ltag}: {adata_raw_by_lineage[ltag].shape}")

## Helper functions (verbatim from `260423-utr_proportions_calculations.ipynb`)


In [ ]:
def pseudobulk_gene_props(adata, gene_name, lineage, celltypes_in_lineage):
    # Pseudobulk UTR proportions per celltype within a lineage with Wilson 95% CI.
    gene_mask = adata.var["gene_name"] == gene_name
    if gene_mask.sum() == 0:
        return None
    sub = adata[adata.obs["lineage"] == lineage, gene_mask]
    sites = sub.var["UTR_site"].values
    utr_sizes = sub.var["utr_size"].values.astype(float)
    order = np.argsort(utr_sizes)
    sites = sites[order]
    utr_sizes = utr_sizes[order]

    X = sub.X
    if sp.issparse(X):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float64)[:, order]

    rows = []
    for ct in celltypes_in_lineage:
        mask = sub.obs["celltype"].values == ct
        if mask.sum() == 0:
            continue
        counts_per_site = X[mask].sum(axis=0)
        total = counts_per_site.sum()
        for j, (site, size) in enumerate(zip(sites, utr_sizes)):
            c = float(counts_per_site[j])
            prop = c / total if total > 0 else 0.0
            ci_lo, ci_hi = (
                proportion_confint(
                    int(round(c)), int(round(total)), alpha=0.05, method="wilson"
                )
                if total > 0
                else (0.0, 0.0)
            )
            rows.append(
                {
                    "gene_name": gene_name,
                    "lineage": lineage,
                    "celltype": ct,
                    "UTR_site": site,
                    "utr_size": size,
                    "count": c,
                    "total": float(total),
                    "proportion": prop,
                    "ci_low": ci_lo,
                    "ci_high": ci_hi,
                }
            )
    return pd.DataFrame(rows)


def compute_W(proportions, utr_sizes):
    # W = sum(proportion_i * rank_i); rank_i in [0, 1] by utr_size.
    K = len(utr_sizes)
    if K > 1:
        ranks = (rankdata(utr_sizes, method="average") - 1) / (K - 1)
    else:
        ranks = np.array([0.5])
    return float(np.dot(proportions, ranks))


def bootstrap_W_ci(
    adata,
    gene_name,
    lineage,
    celltypes_in_lineage,
    n_boot=N_BOOTSTRAP,
    alpha=0.05,
    seed=RANDOM_STATE,
):
    gene_mask = adata.var["gene_name"] == gene_name
    if gene_mask.sum() == 0:
        return None
    sub = adata[adata.obs["lineage"] == lineage, gene_mask]
    utr_sizes = sub.var["utr_size"].values.astype(float)
    order = np.argsort(utr_sizes)
    utr_sizes_sorted = utr_sizes[order]

    X = sub.X
    if sp.issparse(X):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float64)[:, order]
    K = X.shape[1]

    rng = np.random.default_rng(seed)
    rows = []
    for ct in celltypes_in_lineage:
        mask = sub.obs["celltype"].values == ct
        X_ct = X[mask]
        n_cells = X_ct.shape[0]
        if n_cells == 0:
            continue
        n_expressing = int((X_ct.sum(axis=1) > 0).sum())

        counts_obs = X_ct.sum(axis=0)
        total_obs = counts_obs.sum()
        props_obs = counts_obs / total_obs if total_obs > 0 else np.zeros(K)
        W_obs = compute_W(props_obs, utr_sizes_sorted)

        W_boots = np.empty(n_boot)
        for b in range(n_boot):
            idx = rng.choice(n_cells, size=n_cells, replace=True)
            c_b = X_ct[idx].sum(axis=0)
            t_b = c_b.sum()
            p_b = c_b / t_b if t_b > 0 else np.zeros(K)
            W_boots[b] = compute_W(p_b, utr_sizes_sorted)

        ci_lo = np.percentile(W_boots, 100 * alpha / 2)
        ci_hi = np.percentile(W_boots, 100 * (1 - alpha / 2))
        rows.append(
            {
                "gene_name": gene_name,
                "lineage": lineage,
                "celltype": ct,
                "W": W_obs,
                "W_ci_low": ci_lo,
                "W_ci_high": ci_hi,
                "n_expressing": n_expressing,
                "n_cells_total": n_cells,
            }
        )
    return pd.DataFrame(rows)


def W_per_cell(adata, gene_name, lineage):
    gene_mask = adata.var["gene_name"] == gene_name
    if gene_mask.sum() == 0:
        return None
    sub = adata[adata.obs["lineage"] == lineage, gene_mask]
    utr_sizes = sub.var["utr_size"].values.astype(float)
    order = np.argsort(utr_sizes)
    utr_sizes_sorted = utr_sizes[order]
    K = len(utr_sizes_sorted)
    ranks = (
        (rankdata(utr_sizes_sorted, method="average") - 1) / (K - 1)
        if K > 1
        else np.array([0.5])
    )

    X = sub.X
    if sp.issparse(X):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float64)[:, order]
    gene_total = X.sum(axis=1)
    expressing = gene_total > 0

    W_cells = np.zeros(X.shape[0])
    W_cells[expressing] = (X[expressing] @ ranks) / gene_total[expressing]
    return pd.DataFrame(
        {
            "gene_name": gene_name,
            "lineage": lineage,
            "barcode": sub.obs_names,
            "celltype": sub.obs["celltype"].values,
            "pseudotime": sub.obs[f"dpt_pseudotime_{LINEAGE_TAG[lineage]}"].values,
            "W_cell": W_cells,
            "expressing": expressing,
        }
    )


def utr_length_distribution(adata_counts, gene_name, lineage, celltypes_in_lineage):
    gene_mask = adata_counts.var["gene_name"] == gene_name
    if gene_mask.sum() == 0:
        return None
    sub = adata_counts[adata_counts.obs["lineage"] == lineage, gene_mask]
    utr_sizes = sub.var["utr_size"].values.astype(float)

    X = sub.X
    if sp.issparse(X):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float64)

    rows = []
    for ct in celltypes_in_lineage:
        mask = sub.obs["celltype"].values == ct
        if mask.sum() == 0:
            continue
        c_per_feat = X[mask].sum(axis=0)
        for size, c in zip(utr_sizes, c_per_feat):
            if c > 0:
                rows.append(
                    {
                        "gene_name": gene_name,
                        "lineage": lineage,
                        "celltype": ct,
                        "utr_size": size,
                        "count": int(c),
                    }
                )
    return pd.DataFrame(rows)


def consensus_peaks(adata, gene_name):
    gene_mask = adata.var["gene_name"] == gene_name
    if gene_mask.sum() == 0:
        return None
    sv = adata.var[gene_mask].sort_values("utr_size")
    return pd.DataFrame(
        {
            "gene_name": gene_name,
            "UTR_site": sv["UTR_site"].values,
            "utr_size": sv["utr_size"].values,
        }
    )


print("Helpers defined.")

## Loop over target (gene, lineage) pairs


In [ ]:
t0 = time.time()
all_props = []
all_W_cluster = []
all_W_cell = []
all_utrlen = []
all_peaks = []

for lname in LINEAGE_ORDER:
    ltag = LINEAGE_TAG[lname]
    adata_raw = adata_raw_by_lineage[ltag]
    cts = CANONICAL_CELLTYPE_BY_LINEAGE[lname]
    targets = target_genes_by_lineage[ltag]
    print(
        f"\n=== {lname} ({ltag}): {len(targets)} target genes, {len(cts)} celltypes ==="
    )

    # Verify gene_name exists in the h5ad's var
    available = set(adata_raw.var["gene_name"].unique())
    n_missing = sum(1 for g in targets if g not in available)
    if n_missing:
        print(
            f"  WARN: {n_missing}/{len(targets)} target genes not in {ltag} h5ad var.gene_name"
        )

    for j, gene in enumerate(targets):
        if (j + 1) % 20 == 0:
            print(f"  {j+1}/{len(targets)} ({time.time()-t0:.0f}s)")
        if gene not in available:
            continue
        df_p = pseudobulk_gene_props(adata_raw, gene, lname, cts)
        if df_p is not None:
            all_props.append(df_p)
        df_w = bootstrap_W_ci(adata_raw, gene, lname, cts)
        if df_w is not None:
            all_W_cluster.append(df_w)
        df_wc = W_per_cell(adata_raw, gene, lname)
        if df_wc is not None:
            all_W_cell.append(df_wc)
        df_u = utr_length_distribution(adata_raw, gene, lname, cts)
        if df_u is not None:
            all_utrlen.append(df_u)
        df_pk = consensus_peaks(adata_raw, gene)
        if df_pk is not None:
            all_peaks.append(df_pk)

print(f"\nDone in {time.time()-t0:.0f}s")

In [ ]:
def safe_concat(dfs):
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


df_props = safe_concat(all_props)
df_W_cluster = safe_concat(all_W_cluster)
df_W_cell = safe_concat(all_W_cell)
df_utrlen = safe_concat(all_utrlen)
df_peaks = safe_concat(all_peaks).drop_duplicates(subset=["gene_name", "UTR_site"])

print(f"pseudobulk_proportions:    {df_props.shape}")
print(f"W_per_cluster:             {df_W_cluster.shape}")
print(f"W_per_cell:                {df_W_cell.shape}")
print(f"utr_length_distributions:  {df_utrlen.shape}")
print(f"consensus_peak_positions:  {df_peaks.shape}")

## Save CSVs and `gene_info.json`


In [ ]:
df_props.to_csv(RESULTS_DIR / "pseudobulk_proportions.csv", index=False)
df_W_cluster.to_csv(RESULTS_DIR / "W_per_cluster.csv", index=False)
df_W_cell.to_csv(RESULTS_DIR / "W_per_cell.csv", index=False)
df_utrlen.to_csv(RESULTS_DIR / "utr_length_distributions.csv", index=False)
df_peaks.to_csv(RESULTS_DIR / "consensus_peak_positions.csv", index=False)

print("Saved CSVs:")
for n in [
    "pseudobulk_proportions",
    "W_per_cluster",
    "W_per_cell",
    "utr_length_distributions",
    "consensus_peak_positions",
]:
    p = RESULTS_DIR / f"{n}.csv"
    print(f"  {n}.csv  ({p.stat().st_size/1024:.0f} KB)")

In [ ]:
gene_info = {
    "target_genes_by_lineage": target_genes_by_lineage,
    "top_hits_by_lineage": top_hits_by_lineage,
    "mouse_orthologs": MOUSE_ORTHOLOGS_HUMAN,
    "mouse_to_human": MOUSE_TO_HUMAN_ORTHO,
    "asd_genes": ASD_GENES_LI2023,
    "celltype_order_by_lineage": {
        LINEAGE_TAG[l]: cts for l, cts in CANONICAL_CELLTYPE_BY_LINEAGE.items()
    },
    "lineage_order": LINEAGE_ORDER,
    "lineage_slugs": {l: LINEAGE_TAG[l] for l in LINEAGE_ORDER},
    "n_top_hits": N_TOP_HITS,
    "n_bootstrap": N_BOOTSTRAP,
    "random_state": RANDOM_STATE,
    "source_dirmult_dir": str(BASEPATH / "utrcalling/dirmult_results"),
}

gi_path = RESULTS_DIR / "gene_info.json"
with open(gi_path, "w") as f:
    json.dump(gene_info, f, indent=2)
print(f"Saved {gi_path}")

In [ ]:
# Sanity assertions
# Note: (gene_name, UTR_site) is NOT unique in the var (e.g. TMSB15B has two
# distinct polyadenylation sites both labelled "TMSB15B_pA1" but with different
# utr_size). The true per-row unique key includes utr_size as a disambiguator.
# All downstream stats (DirMult LRT, composite ranking) group by ENSG and treat
# each site as a separate isoform indexed by its column in X, so this is a
# pure labelling artifact in the upstream UTR-calling annotation.
assert (
    df_props.duplicated(
        ["gene_name", "lineage", "celltype", "UTR_site", "utr_size"]
    ).sum()
    == 0
)
assert df_W_cluster.duplicated(["gene_name", "lineage", "celltype"]).sum() == 0
assert df_W_cell["W_cell"].between(0, 1).all() or df_W_cell["W_cell"].isna().any()

# Diagnostic: warn about (gene_name, UTR_site) collisions so consumers (e.g.
# stacked-bar plots) can disambiguate legend labels by appending utr_size.
collisions = (
    df_props.groupby(["gene_name", "UTR_site"], observed=True)["utr_size"]
    .nunique()
    .reset_index(name="n_unique_sizes")
)
collisions = collisions[collisions["n_unique_sizes"] > 1]
if len(collisions):
    print(
        f"NOTE: {collisions['gene_name'].nunique()} gene(s) have duplicate "
        f"(gene_name, UTR_site) labels; downstream plots should disambiguate "
        f"by appending utr_size to the legend."
    )
    print(collisions.to_string(index=False))
else:
    print("No (gene_name, UTR_site) collisions detected.")

print("Sanity checks passed.")

In [ ]:
df_props[df_props.duplicated(["gene_name", "lineage", "celltype", "UTR_site"])]

In [ ]:
df_props.query("gene_name=='TMSB15B'")